## **Exercice 1 : Identifier les limites des modèles de langage traditionnels**

Considérons la phrase suivante : « Le scientifique, qui travaillait sur ce projet depuis des années, a finalement fait une découverte capitale.

**Limites des modèles de langage traditionnels**

Les modèles de langage traditionnels traitent les mots de manière séquentielle. Ils peuvent avoir du mal à relier « scientifique » à « découverte » à cause des nombreux mots intermédiaires (tendance à vite oublier les informations).

**Impact sur la compréhension**

Cette limitation peut entraîner une mauvaise compréhension du sens de la phrase, ce qui dégrade les performances pour des tâches comme la réponse aux questions ou le résumé automatique.

**Apport du mécanisme d'attention**

Le mécanisme d'attention permet au modèle de se concentrer directement sur les mots importants, quelle que soit leur position. Ainsi, il relie facilement « scientifique » à « découverte », ce qui améliore la compréhension du contexte et la précision des réponses.

## **Exercice 2 : Explorer l'impact de l'attention dans les transformateurs**

**Tâche choisie :** Réponse aux questions

Les modèles Transformers comme BERT et GPT utilisent le mécanisme d'attention pour identifier les mots les plus pertinents d'un texte en fonction de la question posée. Cela leur permet de comprendre le contexte global plutôt que de traiter les mots uniquement dans leur ordre d'apparition.

Exemples d'apport de l'attention :

* Dépendances à long terme : le modèle relie des mots éloignés dans un texte, ce qui permet de retrouver correctement l'information recherchée.
* Résolution des ambiguïtés : l'attention utilise le contexte pour déterminer le bon sens d'un mot ayant plusieurs significations.
* Contextes complexes : le modèle prend en compte l'ensemble du passage pour produire une réponse précise, même lorsque l'information est répartie dans plusieurs phrases.

**Comparaison avec des modèles sans attention :**

* Sans attention : difficulté à relier des informations éloignées, compréhension limitée des longs textes et réponses moins précises.
* Avec attention : meilleure compréhension du contexte, gestion efficace des dépendances à longue distance et réponses plus exactes.

Le mécanisme d'attention est la principale innovation qui permet aux Transformers d'obtenir des performances nettement supérieures aux modèles traditionnels pour les systèmes de réponse aux questions.

## **Exercice 3 : Création d’un système RAG simple pour répondre aux questions**

### **Source de connaissances pour le système RAG**

Pour cet exercice, nous allons utiliser un texte simple qui fournit des conseils pour l'achat d'un ordinateur portable pour les nouveaux étudiants. Cela nous permettra de simuler un scénario où le modèle doit récupérer des informations pertinentes pour répondre à des questions spécifiques.

In [11]:
knowledge_source = """
Bienvenue dans votre guide pour choisir le meilleur ordinateur portable pour vos études universitaires. Un bon ordinateur portable est essentiel pour la prise de notes, les recherches et la rédaction de travaux. Voici quelques points clés à considérer.

**Budget :** Définissez un budget réaliste. Les ordinateurs portables varient de 150000 à 300000 FCFA ou plus. Pour la plupart des étudiants, un budget de 300000 à 400000 FCFA offre un bon équilibre entre performances et prix.

**Système d'exploitation :** Les options principales sont Windows, macOS et Chrome OS.
*   **Windows :** Le plus polyvalent, compatible avec la plupart des logiciels universitaires. De nombreuses marques proposent des ordinateurs portables Windows.
*   **macOS :** Excellent pour la créativité et la stabilité, mais souvent plus cher. L'écosystème Apple est très intégré.
*   **Chrome OS :** Idéal pour les tâches basées sur le cloud et les budgets serrés. Moins puissant pour les logiciels lourds.

**Spécifications techniques :**
*   **Processeur (CPU) :** Pour une utilisation générale, un Intel Core i5 ou un AMD Ryzen 5 est suffisant. Pour les tâches plus exigeantes (montage vidéo, programmation lourde), visez un Core i7 ou Ryzen 7.
*   **Mémoire vive (RAM) :** 8 Go de RAM est le minimum recommandé. 16 Go est préférable si votre budget le permet, pour le multitâche intensif.
*   **Stockage :** Un SSD (Solid State Drive) de 256 Go est un bon point de départ. 512 Go ou plus est idéal. Évitez les disques durs (HDD) qui sont lents.
*   **Taille de l'écran :** 13 ou 14 pouces sont portables. 15 pouces offre plus de confort visuel mais est moins facile à transporter. 17 pouces est généralement trop grand pour une utilisation étudiante quotidienne.

**Autonomie de la batterie :** Cherchez un ordinateur portable avec au moins 8 heures d'autonomie pour tenir une journée de cours sans recharger.

**Poids et portabilité :** Si vous vous déplacez souvent, un ordinateur léger (moins de 1,5 kg) est un atout.

**Connectivité :** Assurez-vous d'avoir les ports nécessaires (USB-A, USB-C, HDMI). Le Wi-Fi 6 est un plus pour des connexions rapides.

**Marques populaires :** Dell, HP, Lenovo, Apple, Asus et Acer sont des marques réputées pour leurs ordinateurs portables étudiants.

En suivant ces conseils, vous devriez être en mesure de choisir un ordinateur portable qui répondra à vos besoins académiques et à votre budget."""

print("Source de connaissances chargée.")

Source de connaissances chargée.


In [12]:
!pip install transformers torch faiss-cpu -q

Maintenant que les bibliothèques sont installées, nous allons charger un modèle BERT pré-entraîné, segmenter la source de connaissances en fragments, créer des embeddings pour chaque fragment et les stocker dans une base de données FAISS.

In [13]:
from transformers import AutoTokenizer, AutoModel
import torch
import faiss
import numpy as np

# 1. Charger le tokenizer et le modèle pré-entraîné (BERT Multilingue pour le français)
tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")
model = AutoModel.from_pretrained("bert-base-multilingual-cased")

# Fonction pour diviser le texte en fragments
def chunk_text(text, max_len=256):
    sentences = text.split('.')
    chunks = []
    current_chunk = ""
    for sentence in sentences:
        if len(current_chunk) + len(sentence) + 1 < max_len:
            current_chunk += sentence + "."
        else:
            if current_chunk:
                chunks.append(current_chunk.strip())
            current_chunk = sentence + "."
    if current_chunk:
        chunks.append(current_chunk.strip())
    return chunks

# Diviser la source de connaissances en fragments
chunks = chunk_text(knowledge_source)

# 2. Créer des embeddings pour chaque fragment
def get_embeddings(texts):
    inputs = tokenizer(texts, return_tensors='pt', padding=True, truncation=True, max_length=256)
    with torch.no_grad():
        outputs = model(**inputs)
    # Utiliser le vecteur du jeton CLS (premier jeton) comme embedding de phrase
    return outputs.last_hidden_state[:, 0, :].numpy()

chunk_embeddings = get_embeddings(chunks)

# 3. Initialiser FAISS et ajouter les embeddings
dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension) # Utilisation de L2 pour la distance euclidienne
index.add(chunk_embeddings) # Ajout des vecteurs à l'index

print(f"Nombre de fragments: {len(chunks)}")
print(f"Dimensions des embeddings: {dimension}")
print(f"Index FAISS créé avec {index.ntotal} vecteurs.")

# Stocker les chunks pour la récupération
vector_store = {
    "chunks": chunks,
    "index": index
}

print("Embeddings créés et stockés dans FAISS.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Nombre de fragments: 11
Dimensions des embeddings: 768
Index FAISS créé avec 11 vecteurs.
Embeddings créés et stockés dans FAISS.


In [14]:
# 4. Implémenter le système de récupération (Retriever)
def retrieve_documents(query, top_k=3):
    # Obtenir l'embedding de la question
    query_embedding = get_embeddings([query])

    # Rechercher les documents les plus similaires dans l'index FAISS
    D, I = vector_store["index"].search(query_embedding, top_k) # D: distances, I: indices

    # Récupérer les chunks correspondants
    retrieved_chunks = [vector_store["chunks"][i] for i in I[0]]
    return retrieved_chunks

print("Fonction de récupération implémentée.")

# Exemple d'utilisation du retriever
question = "Quel est le budget recommandé pour un ordinateur portable étudiant ?"
results = retrieve_documents(question)
print(f"\nQuestion: {question}")
print("Documents récupérés:")
for i, doc in enumerate(results):
    print(f"- {doc}")

Fonction de récupération implémentée.

Question: Quel est le budget recommandé pour un ordinateur portable étudiant ?
Documents récupérés:
- Bienvenue dans votre guide pour choisir le meilleur ordinateur portable pour vos études universitaires. Un bon ordinateur portable est essentiel pour la prise de notes, les recherches et la rédaction de travaux. Voici quelques points clés à considérer.
- En suivant ces conseils, vous devriez être en mesure de choisir un ordinateur portable qui répondra à vos besoins académiques et à votre budget..
- **Budget :** Définissez un budget réaliste. Les ordinateurs portables varient de 150000 à 300000 FCFA ou plus. Pour la plupart des étudiants, un budget de 300000 à 400000 FCFA offre un bon équilibre entre performances et prix.


Maintenant que nous avons une fonction de récupération, la dernière étape consiste à intégrer un modèle GPT (générateur) pour formuler une réponse cohérente à partir des documents récupérés et de la question de l'utilisateur.

La prochaine étape consiste à implémenter le système de récupération (retriever) qui prendra une question de l'utilisateur, générera son embedding, et récupérera les documents les plus pertinents de la base de données FAISS.

In [15]:
!pip install google-genai -q

Pour utiliser le modèle Gemini, vous devez fournir votre clé API. Pour des raisons de sécurité, il est recommandé de ne pas inclure directement votre clé API dans le code. Vous pouvez la stocker comme une variable d'environnement ou l'entrer lorsque le système le demande. Pour cet exemple, nous utiliserons `input()`.

In [22]:
import google.generativeai as genai

# Pour cet exemple interactif, nous demandons la clé à l'utilisateur
GOOGLE_API_KEY = input("Veuillez entrer votre clé API Google Gemini: ")
genai.configure(api_key=GOOGLE_API_KEY)

# 5. Implémenter le générateur (GPT)
def generate_answer(question, context):
    prompt = f"""En utilisant uniquement les informations suivantes, répondez à la question.
Si les informations ne sont pas suffisantes, indiquez que vous ne pouvez pas répondre avec le contexte donné.

Contexte:
{context}

Question: {question}
Réponse:"""

    try:
        model = genai.GenerativeModel('gemini-2.5-flash')
        response = model.generate_content(prompt)
        return response.text
    except Exception as e:
        return f"Désolé, une erreur est survenue lors de la génération de la réponse: {e}"

print("Fonction de génération de réponse implémentée.")

Veuillez entrer votre clé API Google Gemini: test
Fonction de génération de réponse implémentée.


Nous avons maintenant les composants de récupération et de génération. L'étape finale est de les combiner dans un système RAG complet et de le tester.

In [20]:
# 6. Combiner le Retriever et le Générateur en un système RAG
def rag_system(question, top_k=3):
    # Étape 1: Récupération des documents pertinents
    retrieved_docs = retrieve_documents(question, top_k)

    # Concaténer les documents récupérés pour former le contexte
    context = "\n\n".join(retrieved_docs)

    # Étape 2: Génération de la réponse
    answer = generate_answer(question, context)

    return answer, retrieved_docs

print("Système RAG combiné.")

# 7. Tester le système RAG
questions_to_test = [
    "Quel est le budget recommandé pour un ordinateur portable étudiant ?",
    "Quels sont les systèmes d'exploitation populaires et leurs caractéristiques ?",
    "Quand est-il préférable d'acheter un HDD plutôt qu'un SSD ?" # Question hors contexte
]

for q in questions_to_test:
    print(f"\n--- Question: {q} ---")
    answer, docs = rag_system(q)
    print("Réponse:", answer)
    print("Documents sources:")
    for doc in docs:
        print(f"- {doc}")


Système RAG combiné.

--- Question: Quel est le budget recommandé pour un ordinateur portable étudiant ? ---


Réponse: Désolé, une erreur est survenue lors de la génération de la réponse: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 6.193528013s.
Documents sources:
- Bienvenue dans votre guide pour choisir le meilleur ordinateur portable pour vos études universitaires. Un bon ordinateur portable est essentiel pour la prise de notes, les recherches et la rédaction de travaux. Voici quelques points clés à considérer.
- En suivant ces conseils, vous devriez être en mesure de choisir un ordinateur portable qui répondra à vos b

Réponse: Désolé, une erreur est survenue lors de la génération de la réponse: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 5.466448776s.
Documents sources:
- En suivant ces conseils, vous devriez être en mesure de choisir un ordinateur portable qui répondra à vos besoins académiques et à votre budget..
- Bienvenue dans votre guide pour choisir le meilleur ordinateur portable pour vos études universitaires. Un bon ordinateur portable est essentiel pour la prise de notes, les recherches et la rédaction de travaux. Vo

Réponse: Désolé, une erreur est survenue lors de la génération de la réponse: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 4.730042995s.
Documents sources:
- Bienvenue dans votre guide pour choisir le meilleur ordinateur portable pour vos études universitaires. Un bon ordinateur portable est essentiel pour la prise de notes, les recherches et la rédaction de travaux. Voici quelques points clés à considérer.
- *   **Mémoire vive (RAM) :** 8 Go de RAM est le minimum recommandé. 16 Go est préférable si votre budget le

In [18]:
# 6. Combiner le Retriever et le Générateur en un système RAG
def rag_system(question, top_k=3):
    # Étape 1: Récupération des documents pertinents
    retrieved_docs = retrieve_documents(question, top_k)

    # Concaténer les documents récupérés pour former le contexte
    context = "\n\n".join(retrieved_docs)

    # Étape 2: Génération de la réponse
    answer = generate_answer(question, context)

    return answer, retrieved_docs

print("Système RAG combiné.")

# 7. Tester le système RAG
questions_to_test = [
    "Quel est le budget recommandé pour un ordinateur portable étudiant ?",
    "Quels sont les systèmes d'exploitation populaires et leurs caractéristiques ?",
    "Quand est-il préférable d'acheter un HDD plutôt qu'un SSD ?" # Question hors contexte
]

for q in questions_to_test:
    print(f"\n--- Question: {q} ---")
    answer, docs = rag_system(q)
    print("Réponse:", answer)
    print("Documents sources:")
    for doc in docs:
        print(f"- {doc}")


Système RAG combiné.

--- Question: Quel est le budget recommandé pour un ordinateur portable étudiant ? ---


Réponse: Désolé, une erreur est survenue lors de la génération de la réponse: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 46.311484694s.
Documents sources:
- Bienvenue dans votre guide pour choisir le meilleur ordinateur portable pour vos études universitaires. Un bon ordinateur portable est essentiel pour la prise de notes, les recherches et la rédaction de travaux. Voici quelques points clés à considérer.
- En suivant ces conseils, vous devriez être en mesure de choisir un ordinateur portable qui répondra à vos 

Réponse: Désolé, une erreur est survenue lors de la génération de la réponse: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 45.6497665s.
Documents sources:
- En suivant ces conseils, vous devriez être en mesure de choisir un ordinateur portable qui répondra à vos besoins académiques et à votre budget..
- Bienvenue dans votre guide pour choisir le meilleur ordinateur portable pour vos études universitaires. Un bon ordinateur portable est essentiel pour la prise de notes, les recherches et la rédaction de travaux. Voi

Réponse: Désolé, une erreur est survenue lors de la génération de la réponse: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 44.545627423s.
Documents sources:
- Bienvenue dans votre guide pour choisir le meilleur ordinateur portable pour vos études universitaires. Un bon ordinateur portable est essentiel pour la prise de notes, les recherches et la rédaction de travaux. Voici quelques points clés à considérer.
- *   **Mémoire vive (RAM) :** 8 Go de RAM est le minimum recommandé. 16 Go est préférable si votre budget l

En somme le système RAG a été implémenté avec succès. Les embeddings générés par BERT, la recherche vectorielle avec FAISS et la récupération des documents fonctionnent correctement. La génération des réponses est également satisfaisante lorsque l'API est disponible. Cependant, l'évaluation complète n'a pas pu être menée en raison de la limitation du quota gratuit de l'API Gemini.

## **Exercice 4 : Analyse du rôle de BERT dans les systèmes RAG**

### **Application choisie : Réponse aux questions**

#### **1. Comment BERT est-il utilisé dans le composant de recherche d'un système RAG ?**

Dans un système RAG (Retrieval Augmented Generation) pour la réponse aux questions, BERT joue un rôle fondamental dans le composant de *recherche* (retriever). Son objectif est de convertir à la fois la question de l'utilisateur et les documents de la base de connaissances en représentations vectorielles (embeddings) de haute dimension. Voici comment cela fonctionne :

*   **Encodage des documents :** Chaque fragment (chunk) de texte de la base de connaissances (notre `knowledge_source` dans l'Exercice 3) est passé à travers un modèle BERT pré-entraîné. BERT génère un vecteur numérique (embedding) pour chaque fragment, capturant son sens sémantique. Ces embeddings sont ensuite stockés dans une base de données vectorielle (comme FAISS que nous avons utilisée).
*   **Encodage de la requête :** Lorsqu'une question est posée par l'utilisateur, elle est également encodée par le même modèle BERT pour produire un embedding de la requête.
*   **Recherche de similarité :** Le retriever compare l'embedding de la question avec tous les embeddings des fragments de la base de données vectorielle. Il utilise une mesure de similarité (par exemple, la distance cosinus ou la distance euclidienne comme dans FAISS) pour trouver les fragments de texte les plus pertinents par rapport à la question. Ces fragments pertinents sont ensuite passés au générateur.

#### **2. Avantages des plongements BERT pour la recherche par rapport à d'autres méthodes**

L'utilisation des embeddings BERT pour la recherche offre des avantages significatifs par rapport aux méthodes plus anciennes comme TF-IDF ou les embeddings de mots (Word2Vec, GloVe) :

*   **Compréhension contextuelle :** Contrairement à TF-IDF qui se base sur la fréquence des mots, ou aux embeddings de mots qui attribuent un seul vecteur à chaque mot indépendamment du contexte, BERT est un modèle *contextuel*. Il comprend le sens d'un mot en fonction des mots qui l'entourent dans une phrase. Cela permet aux embeddings BERT de capturer des nuances sémantiques beaucoup plus riches, améliorant la pertinence des documents récupérés.
*   **Gestion des synonymes et paraphrases :** Grâce à sa compréhension contextuelle, BERT peut identifier que des phrases différentes mais sémantiquement similaires (par exemple, une question et un fragment de texte) sont liées, même si elles n'utilisent pas exactement les mêmes mots. Cela est difficile pour TF-IDF et limité pour les embeddings de mots.
*   **Dépendances à long terme :** Les architectures basées sur les Transformers (dont BERT fait partie) sont excellentes pour capturer les dépendances à long terme dans le texte, ce qui signifie qu'elles peuvent relier des informations éloignées dans un long document ou une longue question. Ceci est crucial pour la compréhension de textes complexes.
*   **Performance globale :** En fournissant au générateur des documents contextuellement plus pertinents, les embeddings BERT augmentent considérablement la qualité et la précision des réponses générées par le système RAG dans son ensemble.

#### **3. Impact de la qualité des représentations BERT sur les performances globales du système RAG**

La qualité des représentations (embeddings) générées par BERT a un impact direct et majeur sur les performances globales du système RAG :

*   **Précision de la récupération :** Si les embeddings BERT ne capturent pas adéquatement le sens sémantique des documents et des requêtes (par exemple, si le modèle BERT est mal entraîné ou si les données sont trop spécifiques pour un modèle généraliste), le retriever pourrait échouer à identifier les documents les plus pertinents. Une mauvaise récupération est le maillon faible du RAG : même un générateur parfait ne pourra pas produire une bonne réponse sans le bon contexte.
*   **Réduction des hallucinations :** Des embeddings de haute qualité garantissent que le contexte fourni au générateur est pertinent et factuel. Cela réduit la probabilité que le générateur invente des informations (hallucinations) ou donne des réponses incorrectes basées sur un contexte erroné.
*   **Pertinence de la réponse :** Un retriever efficace, alimenté par de bons embeddings BERT, assure que le générateur reçoit les informations nécessaires pour formuler une réponse précise, complète et directement liée à la question de l'utilisateur.
*   **Robustesse du système :** De bonnes représentations BERT rendent le système RAG plus robuste aux variations de formulation des questions et à la complexité des documents. Le système sera moins sensible aux mots-clés exacts et plus capable de comprendre l'intention derrière la requête.

## **Exercice 5 : Comparaison de RAG avec les modèles génératifs traditionnels**

**Comparaison des architectures**

Modèle génératif traditionnel (GPT) : génère une réponse à partir des connaissances apprises pendant son entraînement.
RAG : combine un système de recherche (Retriever) qui récupère des documents pertinents et un générateur (comme GPT) qui produit une réponse en s'appuyant sur ces documents.

**Forces et faiblesses**

| Critère | Modèle génératif (GPT)| RAG |
| :--- | :---| :--- |
| Exactitude | Bonne, mais peut produire des hallucinations | Plus élevée grâce aux documents récupérés |
| Véracité des faits  | Dépend des connaissances du modèle      | Appuyée sur des sources externes|
| Nouvelles informations | Nécessite un réentraînement | Peut utiliser des documents récemment ajoutés |
| Vitesse| Plus rapide| Légèrement plus lent (étape de récupération) |

**Cas d'utilisation**

RAG est préférable pour les assistants documentaires, les bases de connaissances d'entreprise, la recherche d'informations et les chatbots utilisant des documents récents.
GPT seul est préférable pour la rédaction, la traduction, le résumé, la génération de contenu créatif ou les conversations générales.

**Analyse des compromis**

Le principal avantage de RAG est sa capacité à fournir des réponses plus fiables et basées sur des documents actualisés. En contrepartie, il nécessite une base documentaire et une étape supplémentaire de recherche, ce qui augmente la complexité et le temps de réponse.

Les modèles génératifs traditionnels sont plus simples et rapides à utiliser, mais ils peuvent produire des informations inexactes lorsqu'ils ne disposent pas des connaissances nécessaires ou lorsque les informations ont évolué.

Le choix entre RAG et un modèle génératif traditionnel dépend de l'application. RAG est plus adapté lorsque la précision et l'actualité des informations sont essentielles, tandis qu'un modèle génératif seul convient mieux aux tâches de génération de texte et aux échanges généraux.

## **Exercice 6 : Explorer l’avenir de RAG en PNL**